In [ ]:
from pathlib import Path
import sys
import time

import matplotlib.pyplot as plt
import torch

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.misc import load_config
from src.predict import MPDDOptions, load_predictor


def take_slice(volume, axis, index):
    return volume.select(axis, index)

In [ ]:
PREDICT_CONFIG = ROOT / "config" / "predict.yaml"
SEED = 0

In [ ]:
torch.manual_seed(SEED)

config = load_config(PREDICT_CONFIG)
run_dir = Path(config.pop("run_dir"))
run_dir = run_dir if run_dir.is_absolute() else ROOT / run_dir
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
predictor = load_predictor(run_dir, device=device)
options = MPDDOptions(**config)

start = time.perf_counter()
volume, stats = predictor.predict(options)
elapsed = time.perf_counter() - start

sample_fractions = [round(value, 4) for value in stats["phase_fractions"]]
print(
    f"volume={tuple(volume.shape)} elapsed={elapsed:.1f}s "
    f"sampler={stats['sampler']} fractions={sample_fractions}"
)

In [ ]:
center = options.volume_size // 2
fig, axes = plt.subplots(1, 3, figsize=(10, 3.5), squeeze=False)
for axis_id, axis in enumerate(axes.ravel()):
    axis.imshow(
        take_slice(volume, axis_id, center),
        cmap="gray",
        vmin=0,
        vmax=options.num_phases - 1,
        interpolation="nearest",
    )
    axis.set_title(f"axis {axis_id}")
    axis.axis("off")
plt.tight_layout()